In [ ]:
import pandas as pd
import logomaker
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

import seaborn as sns

In [ ]:
def make_matrices(cluster_num, biome1, biome2, x=50):

    # read in the data_df for the cluster
    data_df_merged_balanced = pd.read_csv(f"RF_results/raw/{cluster_num}_data_df.tsv", sep='\t') #TODO
    data_df_merged_balanced['biome'] = data_df_merged_balanced['biome'].str.replace(":",";")

    temp = data_df_merged_balanced.groupby("ecosystem_subtype").sum().reset_index()
    temp_1 = temp.drop(columns=['img_split','biome','ecosystem','ecosystem_category','ecosystem_type','specific_ecosystem','cluster_rep'])
    temp_1_melted = temp_1.melt(id_vars="ecosystem_subtype", var_name="pos_aa", value_name="count")
    temp_1_melted[["position", "aa"]] = temp_1_melted['pos_aa'].str.split("_",expand=True)
    temp_1_melted['position'] = temp_1_melted['position'].astype(int)

    # create separate matrices for each 
    ecosystem_matrices = {
        ecosys: subdf.pivot(index="position", columns="aa", values="count").fillna(0).astype(int)
        for ecosys, subdf in temp_1_melted.groupby("ecosystem_subtype")
    }

    # Example: access the 'Lake' matrix
    biome1_matrix = ecosystem_matrices[biome1]

    # Example: access the 'Oceanic' matrix
    biome2_matrix = ecosystem_matrices[biome2]

    # Read in feature importance scores
    feat_imp_df = pd.read_csv(f"RF_results/raw/{cluster_num}_Feature_importance.tsv", sep='\t', usecols=[0,2],names=['col_name','feature_importance_vals'],header=0) #TODO Add as input
    top_50_ft_df = feat_imp_df.head(n=x)

    top_50_ft_df[['position','aa']] = top_50_ft_df['col_name'].str.split("_",expand=True)
    top_50_ft_df['position'] = top_50_ft_df['position'].astype(int)

    biome1_matrix_filtered = biome1_matrix.loc[biome1_matrix.index.intersection(top_50_ft_df['position'])]
    biome2_matrix_filtered = biome2_matrix.loc[biome2_matrix.index.intersection(top_50_ft_df['position'])]

    data_df_merged_balanced_1 = data_df_merged_balanced.groupby('ecosystem_subtype').apply(lambda x: (x==1).sum()).T
    data_df_merged_balanced_1
    data_df_merged_balanced_1[f"Proportion_{biome1}"] = (
        data_df_merged_balanced_1[biome1] / data_df_merged_balanced[data_df_merged_balanced['ecosystem_subtype'] == biome1].shape[0] * 100)
    data_df_merged_balanced_1[f"Proportion_{biome2}"] = (
        data_df_merged_balanced_1[biome2] / data_df_merged_balanced[data_df_merged_balanced['ecosystem_subtype'] == biome2].shape[0] * 100)

    print(data_df_merged_balanced_1)
    data_df_merged_balanced_1 = data_df_merged_balanced_1.T

    top_50_ft_df_t = top_50_ft_df.set_index('col_name').T
    top_features = data_df_merged_balanced_1[top_50_ft_df_t.columns]
    top_features
    print(top_features.T.head(n=20))

    top_50_ft_df
    # only proportions
    top_features_melted = top_features.reset_index().melt(id_vars="ecosystem_subtype")
    top_features_melted_1 = top_features_melted.sort_values(['variable','value'],ascending=[True,False]).groupby('variable').head(n=1)
    top_feat_bars = top_features_melted_1.merge(top_50_ft_df,left_on='variable',right_on='col_name')
    print(top_feat_bars)
    return biome1_matrix_filtered, biome2_matrix_filtered, top_feat_bars, top_features


In [ ]:

def plot_seq_logo(biome1, biome2, biome1_matrix_filtered, biome2_matrix_filtered, top_feat_bars):
    # Create the new figure and subplots
    fig, axes = plt.subplots(4, 1, sharex=True, figsize=(20, 5), dpi = 120)

    # Create the marine logo directly on the first subplot
    AA_logo_biome2 = logomaker.Logo(
        biome2_matrix_filtered,
        ax=axes[1],  # Use the first subplot's axes
        font_name='Arial Rounded MT Bold',
        color_scheme='dmslogo_charge',
        stack_order='small_on_top',  # Ensures visibility of larger letters
        width=2.5
    )

    # Create the freshwater logo directly on the second subplot
    AA_logo_biome1 = logomaker.Logo(
        biome1_matrix_filtered*-1,
        ax=axes[2],  # Use the second subplot's axes
        font_name='Arial Rounded MT Bold',
        color_scheme='dmslogo_charge',
        flip_below=False,
        baseline_width = 10,
        stack_order='small_on_top',  # Ensures visibility of larger letters
        width=2.5
    )

    # create marine bars above first seqlogo (axes[0])
    data_m=top_feat_bars.loc[top_feat_bars['ecosystem_subtype'] == biome2]

    axes[0].bar(
        data_m["position"], 
        data_m["feature_importance_vals"], 
        color="#9dcc7e",
        width=2  # Adjust width to match sequence logo spacing
    )

    # create freshwater bars below last seqlogo (axes[3])
    data_f = top_feat_bars.loc[top_feat_bars['ecosystem_subtype'] == biome1]

    axes[3].bar(
        data_f["position"], 
        data_f["feature_importance_vals"], 
        color="#2B8CBF",
        width=2
    )
    axes[3].invert_yaxis()

    for ax in axes:
        ax.set_xticks([0, 500])

    # Ensure Logomaker uses the same ticks
    AA_logo_biome2.ax.set_xticks(biome1_matrix_filtered.index)
    AA_logo_biome1.ax.set_xticks(biome1_matrix_filtered.index)

    axes[-1].set_xticklabels(biome1_matrix_filtered.index, rotation=90, fontsize=40)

    axes[0].tick_params(labelsize=10)
    axes[1].tick_params(labelsize=10)
    axes[2].tick_params(labelsize=10)
    axes[3].tick_params(labelsize=10)
    
    # Example of how you can set ticks to an evenly distributed length but then the important positions are not highlighted
    # MSE = [np.random.normal(0,1,5) for i in range(max(top_feat_bars['position']))]
    # Sim = np.arange(len(MSE))
    # ax.set(xticks=Sim[0::10])


    # Adjust layout
    plt.tight_layout()
    # Display the combined figure
    # plt.show()

    return plt


In [ ]:

def plot(cluster_num,biome1, biome2, x=50):
    """ takes variables from outside script here sends them to internal functions"""
    biome1_matrix_filtered, biome2_matrix_filtered, top_feat_bars,top_features = make_matrices(cluster_num, biome1, biome2, x)
    plt = plot_seq_logo(biome1, biome2, biome1_matrix_filtered, biome2_matrix_filtered, top_feat_bars)
    return plt, top_features


In [ ]:
plt, top_features = plot("cluster_1", "Lake", "Oceanic", 20)

In [ ]:
top_features_T = top_features.T.reset_index(names='pos_AA')
top_features_T['Proportion_Lake'] = top_features_T['Proportion_Lake']*-1
top_features_T_m = pd.melt(top_features_T[['pos_AA','Proportion_Lake','Proportion_Oceanic']],id_vars="pos_AA")
top_features_T_m

In [ ]:
sns.set_theme(style="whitegrid")
my_palette = sns.diverging_palette(h_neg=100, h_pos=200,s=100,l=60, n=2)
min,max = 2, 50
plt.figure(figsize=(6,10))

b = sns.barplot(
    data=top_features_T_m,
    x="value",
    y="pos_AA",
    hue="ecosystem_subtype",
    palette=['#9dcc7e','#2B8CBF'],
    dodge=False,
    width=-.5,  
)

# show the graph
# b.set_yticklabels(top_200_features_melted.variable)
b.set_xlabel("Amino acid",fontsize=15)
b.set_ylabel("count_of_amino acids in by biome",fontsize=15)

plt.show()